# 02 · Four OCR Contracts and an Honest MTP Benchmark

**Hardware**: 🟡 8–12GB+ for the 0.9B BF16 checkpoint, or two OpenAI-compatible server endpoints. CPU is possible but slow.

This notebook has two deliberately separate halves:

1. `transformers.generate()` establishes the autoregressive correctness baseline for text, table, formula, and KIE.
2. vLLM/SGLang endpoints measure MTP/speculative serving. A normal Transformers call is **not** labelled MTP.

## 0. Setup

In [ ]:
# %pip install "transformers>=5.14,<5.15" accelerate torch pillow requests

import base64
import io
import json
import os
import re
import statistics
import time

import requests
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import AutoProcessor, GlmOcrForConditionalGeneration

MODEL_ID = "zai-org/GLM-OCR"
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

## 1. Make a small document with four kinds of structure

A synthetic page is not a benchmark; it is a controlled smoke test whose reference is known. Replace `image` with a scan after the code path works.

In [ ]:
def get_font(size=28):
    for name in ("DejaVuSans.ttf", "Arial.ttf"):
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    return ImageFont.load_default()

image = Image.new("RGB", (1400, 1000), "white")
draw = ImageDraw.Draw(image)
title, body = get_font(42), get_font(27)
draw.text((60, 45), "INVOICE A-1048", fill="black", font=title)
draw.text((60, 115), "Date: 2026-08-17     Customer: Ada Lovelace", fill="black", font=body)
draw.text((60, 165), "Ship to: 12 Analytical Engine Way", fill="black", font=body)

x = [60, 700, 980, 1280]
y = [250, 315, 380, 445, 510]
for xx in x: draw.line((xx, y[0], xx, y[-1]), fill="black", width=3)
for yy in y: draw.line((x[0], yy, x[-1], yy), fill="black", width=3)
rows = [
    ("Item", "Qty", "Price"),
    ("Brass gear", "2", "$19.95"),
    ("Punch card", "10", "$1.20"),
    ("Total", "", "$51.90"),
]
for row_i, row in enumerate(rows):
    for col_i, value in enumerate(row):
        draw.text((x[col_i] + 14, y[row_i] + 14), value, fill="black", font=body)

draw.text((60, 610), "Formula note: E = m c²", fill="black", font=get_font(34))
draw.text((60, 680), "Payment terms: Net 30. Tax: $0.00", fill="black", font=body)
image

## 2. Load the base model

The processor is shared with the GLM-V family (`Glm46VProcessor` in the checkpoint metadata); the model class is GLM-OCR-specific. `device_map="auto"` requires `accelerate`.

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = GlmOcrForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    dtype="auto",
).eval()
print(type(processor).__name__, type(model).__name__)

In [ ]:
def run_base_model(prompt, max_new_tokens=1024):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    start = time.perf_counter()
    with torch.inference_mode():
        ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    elapsed = time.perf_counter() - start
    new_ids = ids[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(new_ids, skip_special_tokens=True)[0]
    return {"text": text, "tokens": int(new_ids.shape[1]), "seconds": elapsed}

results = {}

## 3. The four contracts

The first three prompts are the predefined training prompts. KIE instead names an exact schema and the missing-value policy.

In [ ]:
prompts = {
    "text": "Text Recognition:",
    "table": "Table Recognition:",
    "formula": "Formula Recognition:",
    "kie": '''Extract this invoice as strict JSON with exactly this schema:
{
  "invoice_id": string | null,
  "date": string | null,
  "customer": string | null,
  "items": [{"description": string, "quantity": number, "price": number}],
  "total": number | null,
  "tax": number | null
}
Use null for missing scalar fields. Do not invent keys. Return JSON only.''',
}

for name, prompt in prompts.items():
    results[name] = run_base_model(prompt)
    print(f"\n--- {name}: {results[name]['tokens']} tokens in {results[name]['seconds']:.2f}s ---")
    print(results[name]["text"][:2500])

## 4. Validate the contracts

Fluent output is not enough. These validators are intentionally cheap: they belong in every regression run, even before a full benchmark such as TEDS or CDM.

In [ ]:
def strip_fence(text):
    text = text.strip()
    return re.sub(r"^```(?:json|html|latex)?\s*|\s*```$", "", text, flags=re.I | re.S).strip()

def json_contract(text, allowed_keys):
    try:
        value = json.loads(strip_fence(text))
    except json.JSONDecodeError as exc:
        return {"valid": False, "error": str(exc)}
    extra = sorted(set(value) - set(allowed_keys)) if isinstance(value, dict) else ["<not an object>"]
    return {"valid": isinstance(value, dict) and not extra, "extra_keys": extra, "value": value}

def paired_tag_counts(text, tags=("table", "tr", "td", "th")):
    return {
        tag: (len(re.findall(fr"<{tag}(?:\s[^>]*)?>", text, re.I)), len(re.findall(fr"</{tag}>", text, re.I)))
        for tag in tags
    }

kie_check = json_contract(results["kie"]["text"], ["invoice_id", "date", "customer", "items", "total", "tax"])
table_tags = paired_tag_counts(results["table"]["text"])
print("KIE:", kie_check)
print("table open/close counts:", table_tags)

## 5. MTP belongs to the serving experiment

Launch the same checkpoint once without speculative decoding and once with the official MTP option. Run the servers sequentially if one GPU cannot hold both. Save each endpoint URL in `AR_URL` and `MTP_URL`.

```bash
# autoregressive target baseline
vllm serve zai-org/GLM-OCR --port 8081 --served-model-name glm-ocr

# MTP/speculative path
vllm serve zai-org/GLM-OCR --port 8082 --served-model-name glm-ocr \
  --speculative-config '{"method":"mtp","num_speculative_tokens":3}'
```

A production benchmark should use a page set and concurrency sweep. The small loop below is a latency sanity check.

In [ ]:
def image_data_uri(pil_image):
    buffer = io.BytesIO()
    pil_image.save(buffer, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode()

def call_server(url, prompt="Text Recognition:", timeout=180):
    payload = {
        "model": "glm-ocr",
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_data_uri(image)}},
                {"type": "text", "text": prompt},
            ],
        }],
        "temperature": 0,
        "max_tokens": 1024,
    }
    start = time.perf_counter()
    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    elapsed = time.perf_counter() - start
    data = response.json()
    usage = data.get("usage", {})
    text = data["choices"][0]["message"]["content"]
    return {"seconds": elapsed, "tokens": usage.get("completion_tokens"), "text": text}

def benchmark(url, warmup=1, repeats=3):
    for _ in range(warmup):
        call_server(url)
    runs = [call_server(url) for _ in range(repeats)]
    return {
        "median_seconds": statistics.median(r["seconds"] for r in runs),
        "tokens_per_second": [r["tokens"] / r["seconds"] if r["tokens"] else None for r in runs],
        "outputs": [r["text"] for r in runs],
    }

In [ ]:
AR_URL = os.getenv("AR_URL")      # e.g. http://127.0.0.1:8081/v1/chat/completions
MTP_URL = os.getenv("MTP_URL")    # e.g. http://127.0.0.1:8082/v1/chat/completions

if AR_URL and MTP_URL:
    server_results = {"ar": benchmark(AR_URL), "mtp": benchmark(MTP_URL)}
    for name, result in server_results.items():
        print(name, result["median_seconds"], result["tokens_per_second"])
    exact_agreement = server_results["ar"]["outputs"][0] == server_results["mtp"]["outputs"][0]
    print("first-output exact agreement:", exact_agreement)
else:
    print("Set AR_URL and MTP_URL after launching the two explicit serving modes.")

## Exercises

1. Build ten page images with increasing font sizes and plot normalized edit distance against visual-token count.
2. Make the KIE schema ambiguous about missing values, then repair it. Does JSON validity change, field accuracy, or both?
3. Benchmark `num_speculative_tokens` 1, 2, 3, and 4. Faster is not guaranteed when acceptance falls.
4. Compare output equality after normalising whitespace separately from byte-for-byte equality. Which contract needs which standard?
5. Add digit precision/recall. Why can whole-text edit distance hide the most expensive error?